In [2]:
import os
from dotenv import load_dotenv
load_dotenv() ## loading all environment variables from .env file

groq_api_key = os.getenv("GROQ_API_KEY")


In [3]:
from langchain_groq import ChatGroq
model=ChatGroq(model="llama-3.1-8b-instant",groq_api_key=groq_api_key)
model

/opt/anaconda3/envs/langchain310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x120a31420>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x120a31330>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [4]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi, My name is Krunal. I am ML engineer.")])

AIMessage(content="Nice to meet you, Krunal. It's great to know that you're an ML engineer. What kind of projects or areas are you currently working on or interested in? Are you into computer vision, NLP, or something else?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 49, 'total_tokens': 99, 'completion_time': 0.093487802, 'completion_tokens_details': None, 'prompt_time': 0.004087635, 'prompt_tokens_details': None, 'queue_time': 0.009185233, 'total_time': 0.097575437}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_020e283281', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b6b4c-ac9d-7ba1-9d9a-20d243785b17-0', usage_metadata={'input_tokens': 49, 'output_tokens': 50, 'total_tokens': 99})

In [5]:
from langchain_core.messages import  AIMessage
model.invoke(
    [
        HumanMessage(content="Hi, My name is Krunal. I am ML engineer."),
        AIMessage(content="Hello Krunal, nice to meet you! How can I assist you today?"),
        HumanMessage(content="Hey what is my name and which field I work in?"),
    ]
)



AIMessage(content='Your name is Krunal, and you work as a Machine Learning (ML) engineer.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 88, 'total_tokens': 109, 'completion_time': 0.021961107, 'completion_tokens_details': None, 'prompt_time': 0.007498591, 'prompt_tokens_details': None, 'queue_time': 0.070808048, 'total_time': 0.029459698}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_6c980774ec', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b6b4c-addb-7872-a30f-a01e759c83a8-0', usage_metadata={'input_tokens': 88, 'output_tokens': 21, 'total_tokens': 109})

## Message History 

We can use massage history class to wrap our model and make it stateful. This will keep track of input and output of the model, and store them in some datastore. Future intractions will then load those messages and pass them into chain as part of the input. Let's see how to use this! 

In [6]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import  BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id: str)-> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(model,get_session_history)


In [7]:
config={"configurable":{"session_id":"chat1"}}

In [8]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi, My name is Krunal. I am ML engineer.")],
    config=config
)

In [9]:
response.content

'Nice to meet you, Krunal. As a Machine Learning (ML) engineer, you must be working on some exciting projects. What specific areas of ML are you interested in or working with, such as computer vision, natural language processing, or reinforcement learning?'

In [10]:
# changing the session_id
config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hello, what is my name?")],
    config=config1
)

In [11]:
response.content

"I'm happy to chat with you, but I don't know your name. I'm a large language model, I don't have the ability to retain information about individual users or their personal details. Each time you interact with me, it's a new conversation and I don't retain any information from previous conversations. Would you like to introduce yourself?"

In [12]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi, My name is John")],
    config=config
)
response.content

"Hi John. It seems like we had a conversation started, and then it was interrupted. I was talking to Krunal, an ML engineer. If you're here to start a new conversation or continue with Krunal's, feel free to let me know. What brings you here today?"

## Prompt Templates

Prompt Templates help to turn raw user infromation into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that bit more complicated. First, let's add in a system message with some custom instructios(but still taking messgaes as input). Next, we'll add in more input besides just the messages.


In [13]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt=ChatPromptTemplate.from_messages(
    [
        ("system","Ypu are a helpful assistant. Answer all the question to the best of your ability."),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [14]:
chain.invoke({"messages":[HumanMessage(content="Hi, My name is Krunal. I am ML engineer.")]})

AIMessage(content="Nice to meet you, Krunal! It's great to connect with another ML engineer. What specific areas of machine learning are you interested in or working on? Are you dealing with computer vision, natural language processing, or perhaps reinforcement learning? I'm here to help with any questions or topics you'd like to discuss.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 66, 'prompt_tokens': 67, 'total_tokens': 133, 'completion_time': 0.092396355, 'completion_tokens_details': None, 'prompt_time': 0.003665903, 'prompt_tokens_details': None, 'queue_time': 0.066334923, 'total_time': 0.096062258}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_9ca2574dca', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b6b4c-bafc-7c12-a41c-1801a665c510-0', usage_metadata={'input_tokens': 67, 'output_tokens': 66, 'total_tokens': 133})

In [15]:
with_message_history= RunnableWithMessageHistory(chain,get_session_history)

In [16]:
config={"configurable":{"session_id":"chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hi, My name is Krunal. I am ML engineer.")],
    config=config
)

In [17]:
response.content

"Nice to meet you, Krunal! It's great to hear that you're an ML engineer. What kind of projects or areas are you currently working on or interested in? I'm here to help with any questions or topics you'd like to discuss. How's your experience in ML engineering so far?"

In [18]:
## Add more complexity to the prompt


prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant. Answer all the question to the best of your ability in {language}."),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [19]:
response= chain.invoke({"messages":[HumanMessage(content="Hi, My name is Krunal. I am ML engineer.")],"language":"Hindi"})
response.content

'नमस्ते क्रुनाल जी, मैं आपकी सहायता करने के लिए यहाँ हूँ। मैं जानता हूँ कि आप एक एमएल इंजीनियर हैं, क्या आप किसी विशिष्ट समस्या या प्रश्न के बारे में पूछना चाहते हैं?'

 Lets now wrap this more complicated chain in a message history class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [20]:
with_message_history= RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
    )

In [21]:
config={"configurable":{"session_id":"chat4"}}
response=with_message_history.invoke(
    {'messages': [HumanMessage(content="Hi, My name is Krunal. I am ML engineer.")] , 'language':'Hindi'},
    config=config
)
response.content

'नमस्ते क्रुणाल जी, मैं आपकी सहायता करने के लिए यहाँ हूँ। आप एक एमएल इंजीनियर हैं, यह बहुत अच्छा है! क्या आप किसी विशिष्ट समस्या या प्रोजेक्ट पर काम कर रहे हैं या आपको कोई विशिष्ट जानकारी चाहिए? मैं आपकी सहायता करने के लिए तैयार हूँ।'

## Managing the Conversation History

one important concept to understand when building chatboats is how to manage conversation history. If left unmanaged, the list of message will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in. 

trim_messages is a utility that reduces a list of chat messages to fit within a specified token limit by selectively keeping or discarding messages (usually older ones), while optionally preserving important messages like system instructions.

In [23]:
from langchain_core.messages import SystemMessage, trim_messages
trimmer = trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)

messages=[
    SystemMessage(content="You are a good assistant."),
    HumanMessage(content="Hi, My name is Krunal. I am ML engineer."),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice-cream."),
    AIMessage(content="Nice."),
    HumanMessage(content="Whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="Thanks!"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes")
    
]

trimmer.invoke(messages)

[SystemMessage(content='You are a good assistant.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Thanks!', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes', additional_kwargs={}, response_metadata={})]

In [27]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain = (
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    | prompt
    | model
)

response=chain.invoke(
    {"messages":messages + [HumanMessage(content="Which ice-cream is the best?")],
    "language":"English"}
)

response.content

"That's a tough question. The best ice cream flavor is subjective and depends on personal preferences. Some popular flavors include:\n\n- Chocolate\n- Vanilla\n- Strawberry\n- Mint Chocolate Chip\n- Cookies and Cream\n\nWhat's your favorite ice cream flavor?"

In [28]:
## Wrap up this in Message History
with_message_history= RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
    )
config={"configurable":{"session_id":"chat5"}}